In [1]:
# ════════════════════════════════════════════════════════════════════
# Bootstrap — clone the Lamahat repo (single source of truth).
# Code, fonts/ and resources/ arrive together at ONE commit, so this
# notebook and the Streamlit app can never drift apart.  Pin BRANCH to
# a feature branch to test unreleased work; leave "main" for releases.
# ════════════════════════════════════════════════════════════════════
REPO   = "https://github.com/abdoljh/Lamahat.git"
BRANCH = "main"

import os, shutil
if os.path.isdir("/content/Lamahat"):
    shutil.rmtree("/content/Lamahat")
!git clone --depth 1 --branch {BRANCH} {REPO} /content/Lamahat
%cd /content/Lamahat
!git log -1 --pretty="✅ Running at commit: %h  %s"


Cloning into '/content/Lamahat'...
remote: Enumerating objects: 182, done.
remote: Counting objects: 100% (182/182), done.
remote: Compressing objects: 100% (156/156), done.
remote: Total 182 (delta 17), reused 105 (delta 14), pack-reused 0 (from 0)
Receiving objects: 100% (182/182), 69.39 MiB | 17.39 MiB/s, done.
Resolving deltas: 100% (17/17), done.
/content/Lamahat
✅ Running at commit: 00ad306  Merge pull request #47 from abdoljh/claude/phase3-review-plan-8sbeue


In [2]:
# ════════════════════════════════════════════════════════════════════
# Settings — edit before running.
# ════════════════════════════════════════════════════════════════════
SOURCE = "drive"           # "zip" (upload plan + review/) or "drive"
DRIVE_SOURCE_DIR = "/content/drive/MyDrive/_Phase3/sources"
ARCHIVE_ZIP_NAME = "Archive.zip"

# ── Book Title & Main Character  ────────────────────────────────────
BOOK_TITLE     = "مذكرات جعفر العسكري"
CHARACTER_NAME = "Jafar al-Askari"

# ── Look ────────────────────────────────────────────────────────────
BOOK_COVER_PICK   = 1          # 1..N over resources/book_cover/
BOOK_COVER_FIT    = "contain"  # fill | contain | blur_pad
BOOK_COVER_ALIGN  = "right"    # center | left | right
TYPOGRAPHY_FAMILY = "B"        # A | B | C
GRADE             = "neutral"  # warm | cool | neutral | bw
CAPTION_BACKPLATE = "off"      # off | subtle | solid
TEXT_SCRIM        = "auto"     # auto | off | soft | band  (auto = plate only on bright/busy frames)
OVERLAY_ANCHOR    = "auto"     # auto | center | lower  (auto = quotes/names lower-third, section marks centered)
TITLE_SUBTITLE    = ""         # optional sub-line under the main title (author / dates); "" = title only
WORD_REVEAL       = True      # word-by-word reveal on over-image quotes (experimental)
PHOTO_BANK_MAX_USES = 2        # how many shots one curated photo may cover (2 spreads the bank further)
GRADE_MAP         = ""         # optional per-section grading JSON path, e.g. "resources/grade_map.json"
                               #   {"opening":"neutral","point_2":"cool","closing":"warm"} — unmapped → GRADE
MUSIC_DB          = -13.0      # music bed level in dB (default -18)

# ── Text styling (blank = pipeline default) ─────────────────────────
TITLE_SIZE    = 1.0   # main-title size multiplier (1.2 = 20% larger)
TITLE_COLOR   = ""    # "#RRGGBB" or "" -> family default (aged gold)
CAPTION_SIZE  = 1.0   # caption size multiplier
CAPTION_COLOR = ""    # "#RRGGBB" or "" -> white
CAPTION_POS   = ""    # fraction of height from bottom, e.g. "0.08"; "" -> default
# NOTE: captions are disabled below via --no-captions; the CAPTION_* knobs
# take effect only if you remove that flag in the render cell.

# ── Output Files & Directories ──────────────────────────────────────
OUTPUT_BASE_DIR         = "output"
OUTPUT_FILE             = f"{OUTPUT_BASE_DIR}/final_cut_{TYPOGRAPHY_FAMILY}.mp4"
LOG_FILE                = f"{OUTPUT_BASE_DIR}/render.log"
CONDITIONED_ZIP_FILE    = f"{OUTPUT_BASE_DIR}/conditioned.zip"
FINAL_ZIP_FILE          = "output_files.zip"
RO_ZIP_FILE             = "output_files_ro.zip"
DRIVE_SAVE_DIR          = "/content/drive/MyDrive/_Phase3/output"
DRIVE_SAVE_RO_DIR       = "/content/drive/MyDrive/_Phase3/output/ro"

# Generated artifacts — supplied at render time (upload .zip or Drive):
PLAN_FILE   = f"{OUTPUT_BASE_DIR}/re_generated_plan.json"
REVIEW_DIR  = f"{OUTPUT_BASE_DIR}/review"
REVIEW_FILE = f"{OUTPUT_BASE_DIR}/review.zip"

# Committed inputs
SCRIPT_FILE = "resources/script/main_script.txt"
AUDIO_FILE  = "resources/audio/narration.mp3"
MUSIC_BED   = "resources/audio/bg_music.mp3"

# print(f"SOURCE={SOURCE!r}  OUTPUT={OUTPUT_FILE!r}  GRADE={GRADE!r}  SCRIM={TEXT_SCRIM!r}")
print(f"OUTPUT= {OUTPUT_FILE}\nGRADE = {GRADE}\nSCRIM = {TEXT_SCRIM}\n")
# ── Optional: host big/private assets on Drive instead of the repo ──
# (pools: character/, book_cover/, photo_bank/ are discovered there)
# import os; os.environ["LAMAHAT_RESOURCES"] = "/content/drive/MyDrive/Lamahat/resources"


OUTPUT= output/final_cut_B.mp4
GRADE = neutral
SCRIM = auto



In [3]:
# Colab-only dependencies (whisperx, openai-whisper, anthropic, Arabic
# shaping).  Streamlit Cloud installs requirements.txt; this file adds
# only the Colab extras on top of Colab's preinstalled stack.
!pip install -q -r requirements-colab.txt
print("✅ Colab dependencies installed!")
!ffmpeg -version 2>&1 | head -1


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 803.2/803.2 kB 17.5 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 7.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 4.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 956.9/956.9 kB 48.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 296.2/296.2 kB 30.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.5/16.5 MB 73.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 39.5/39.5 MB 25.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 69.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 42.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 123.9 MB/

In [ ]:
# OPTIONAL
# Check resources availability
from pathlib import Path
import os

resources_path = Path("resources").resolve()  # repo clone cwd

character_path = resources_path / "character"
book_cover_path = resources_path / "book_cover"

# Display contents of resources/character
print(f"Contents of {character_path}:")
if character_path.exists() and character_path.is_dir():
    chars = sorted(os.listdir(character_path))
    print(f"  Number of items: {len(chars)}")
    for item in chars:
        print(f"    - {item}")
else:
    print("  Directory not found or is not a directory.")

print("\n")

# Display contents of resources/book_cover
print(f"Contents of {book_cover_path}:")
if book_cover_path.exists() and book_cover_path.is_dir():
    covers = sorted(os.listdir(book_cover_path))
    print(f"  Number of items: {len(covers)}")
    for item in covers:
        print(f"    - {item}")
else:
    print("  Directory not found or is not a directory.")


In [ ]:
# OPTIONAL
# Smoke rendering
from pathlib import Path
from phase3.typography_common import TypographySpec
from phase3.typography import render
import os

# Ensure the target directory exists
output_dir = '/content/temp'
os.makedirs(output_dir, exist_ok=True)

rendered_files_expected = []
for fam in ("A", "B", "C"):
    for tpl in ("title_card", "section_mark", "pull_quote", "name_reveal", "date_stamp"):
        spec = TypographySpec(
            template=tpl, family=fam,
            #text="مذكرات جعفر العسكري",
            text = "مُذَكِّراتُ جَعْفَرِ العَسْكَرِيِّ",
            subtitle="١٨٨٥ - ١٩٣٦",
            width=1920, height=1080,
        )
        file_path = Path(f"{output_dir}/typo_{fam}_{tpl}.png")
        render(spec, file_path)
        rendered_files_expected.append(file_path)

print("💨 Typography examples rendering process completed.")

# Verification step
print(f"\nVerifying files in '{output_dir}':")
found_files_actual = []
for f_expected in rendered_files_expected:
    if f_expected.exists():
        found_files_actual.append(f_expected.name)
        print(f"  ✅ Found: {f_expected.name}")
    else:
        print(f"  ❌ NOT found: {f_expected.name}")

if not found_files_actual:
    print(f"No rendered image files were found in '{output_dir}'.")
    print("This suggests an issue with the `render` function from `phase3.typography` not creating the files as expected, or writing them to a different location.")
else:
    print(f"\nSuccessfully found {len(found_files_actual)} rendered files.")


In [ ]:
# OPTIONAL
# Zip typography images
import os
import zipfile

image_dir = '/content/temp'
zip_filename = 'typography_images.zip'

# Create the zip archive
with zipfile.ZipFile(zip_filename, 'w', zipfile.ZIP_DEFLATED) as zipf:
    files_added = []
    if os.path.exists(image_dir):
        for root, dirs, files in os.walk(image_dir):
            for file in files:
                if file.endswith('.png'): # Only zip PNG images
                    file_path = os.path.join(root, file)
                    # Add file to zip, preserving directory structure relative to 'image_dir'
                    zipf.write(file_path, os.path.relpath(file_path, image_dir))
                    files_added.append(file_path)

print(f"🤐 Successfully created '{zip_filename}' containing:")
if files_added:
    for f in files_added:
        print(f"  - {f}")
else:
    print(f"  (No .png files found in '{image_dir}' to zip.)")

In [ ]:
# render_plan help (OPTIONAL)
# !python render_plan.py --help | grep -A2 caption-backplate

In [ ]:
# Verify font discovery (OPTIONAL)
# !python verify_font_discovery.py

In [ ]:
# Check font paths (OPTIONAL)
# !python -c "from phase3.typography import FONT_PATHS; print(FONT_PATHS)"
# print("✅ Pre-testingvthe discovery of Amiri fonts completed!")

In [4]:
# Install anthropic
!pip install anthropic --quiet
print("✴️ Anthropic installed!")

✴️ Anthropic installed!


In [ ]:
# !pip install arabic-reshaper python-bidi
# print("✅ Arabic_reshaper and python-bidi installed!")

In [5]:
# Colab API Keys
print("Retrieving the Anthropic, Pexels & Hugging Face API keys & token ...")
from google.colab import userdata
import os

# Retrieve the Anthropic API key from Colab Secrets
anthropic_api_key = userdata.get('ANTHROPIC_API_KEY')
pexels_api_key = userdata.get('PEXELS_API_KEY')
hf_token = userdata.get('HF_TOKEN')

# Set it as an environment variable for phase3_run.py to use
if anthropic_api_key:
    os.environ['ANTHROPIC_API_KEY'] = anthropic_api_key
    print("🔑 Anthropic API key loaded.")
else:
    print("❌ Warning: ANTHROPIC_API_KEY not found in Colab Secrets. Please ensure it's set correctly.")

if pexels_api_key:
    os.environ['PEXELS_API_KEY'] = pexels_api_key
    print("🔑 Pexels API key loaded.")
else:
    print("❌ Warning: PEXELS_API_KEY not found in Colab Secrets. Please ensure it's set correctly.")

if hf_token:
    os.environ['HF_TOKEN'] = hf_token
    print("🔑 Hugging Face token loaded.")
else:
    print("❌ Warning: HF_TOKEN not found in Colab Secrets. Please ensure it's set correctly.")

Retrieving the Anthropic, Pexels & Hugging Face API keys & token ...
🔑 Anthropic API key loaded.
🔑 Pexels API key loaded.
🔑 Hugging Face token loaded.


In [6]:
# Confirm alignment works (interpolation backend — no install needed)
!python phase3_run.py \
    --script {SCRIPT_FILE} \
    --audio  {AUDIO_FILE} \
    --align-only \
    --align-backend interpolated

print("✅ Confirming alignment completed!")


Script : resources/script/main_script.txt  (3,955 chars)
INFO  phase3.typography_common  Amiri fonts loaded via repo fonts/ (next to phase3): /content/Lamahat/fonts
INFO  phase3.parser  Section detection: 4 boundaries (point_1, point_2, point_3, closing)

Script : 652 word tokens, 5 sections
Audio  : resources/audio/narration.mp3
Total  : 418.0 s
Backend: interpolated

── Running alignment ──────────────────────────────────────────────
WARNING  phase3.align  Using interpolated timings — character-rate estimates only. For real word-level accuracy, install whisperx (pip install whisperx) or whisper (pip install openai-whisper).

  Aligned 652 words in 0.0 s
  Backend used: interpolated

── First 30 word timings ──────────────────────────────────────────
     0.00s →    0.78s   (0.78s)   مذكرات
     0.78s →    1.30s   (0.52s)   جعفر
     1.30s →    2.21s   (0.91s)   العسكري
     2.21s →    2.60s   (0.39s)   رجل
     2.60s →    3.12s   (0.52s)   واحد
     3.12s →    3.51s   (0.39s)   حمل


In [7]:
# Regenerate the plan with the fixes
# Produces output/re_generated_plan.json
!python phase3_run.py \
    --script         {SCRIPT_FILE} \
    --audio          {AUDIO_FILE} \
    --book-title     "{BOOK_TITLE}" \
    --character-name "{CHARACTER_NAME}" \
    --plan-only \
    --save-plan      {PLAN_FILE}

print("✅ Regenerating the plan completed!")


Script : resources/script/main_script.txt  (3,955 chars)
INFO  phase3.typography_common  Amiri fonts loaded via repo fonts/ (next to phase3): /content/Lamahat/fonts
INFO  phase3.parser  Section detection: 4 boundaries (point_1, point_2, point_3, closing)

Script : 652 word tokens, 5 sections
Audio  : resources/audio/narration.mp3
Total  : 418.0 s

── Step 1: Forced alignment ───────────────────────────────────────
2026-07-06 03:52:17.599348: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
INFO  numexpr.utils  NumExpr defaulting to 2 threads.
vocabulary.txt: 0.00B [00:00, ?B/s]
config.json: 2.37kB [00:00, 11.5MB/s]

vocabulary.txt: 460kB [00:00, 23.8MB/s]
tokenizer.json: 2.20MB [00:00, 76.8MB/s]
model.bin: 100% 484M/484M [00:03<00:00, 145MB/s]
20

In [8]:
# Audit the regenerated plan (OPTIONAL)
!python audit_plan.py {PLAN_FILE}
# Expect: ~43 shots, <10% auto-split
# Add --review-dir {REVIEW_DIR} AFTER prebuild to get the effective-holds
# report with dossier-resolved duplicate detection (perceived pacing).

print("✅ Audit completed!")

PLAN AUDIT

Total shots:        85
Plan timeline:      0.13s → 418.04s (417.9s)
Average shot:       4.92s
Range:              2.68s – 7.11s

✓  No gaps or overlaps

Effective visual holds (overlay merged; duplicates via query-equality approximation):
   Effective segments: 55 (plan shots: 85)
   Effective avg hold: 7.60s vs plan avg 4.92s
   ⚠ 18 continuous-footage span(s) exceed 10s:
       10.1s  shots 9–10     (image ×1, overlay ×1)
       12.7s  shots 11–13    (image ×1, overlay ×2)
       11.6s  shots 22–23    (image ×1, overlay ×1)
       10.1s  shots 26–27    (image ×1, overlay ×1)
       11.1s  shots 28–29    (image ×1, overlay ×1)
       10.8s  shots 39–40    (image ×1, overlay ×1)
       11.5s  shots 42–43    (image ×1, overlay ×1)
       12.2s  shots 45–46    (image ×1, overlay ×1)
       11.0s  shots 48–49    (image ×1, overlay ×1)
       10.4s  shots 59–60    (image ×1, overlay ×1)
       15.8s  shots 62–64    (image ×1, overlay ×2)
       11.1s  shots 66–67    (image ×1, 

In [ ]:
# Trimming book cover (OPTIONAL)
# !python trim_book_cover.py overrides/book_cover/my_book.jpg

In [9]:
import os
from pathlib import Path

# Ensure the parent directory for OUTPUT_FILE exists
Path(OUTPUT_FILE).parent.mkdir(parents=True, exist_ok=True)

# Construct the command using an f-string for proper variable interpolation
# Note: $ANTHROPIC_API_KEY and $PEXELS_API_KEY are environment variables,
# and are correctly handled by the shell command, so they don't need Python interpolation.
command = f"""
python prebuild_assets.py \
    --plan           \"{PLAN_FILE}\" \
    --script         \"{SCRIPT_FILE}\" \
    --parallax \
    --book-title     \"{BOOK_TITLE}\" \
    --character-name \"{CHARACTER_NAME}\" \
    --anthropic-key  \"$ANTHROPIC_API_KEY\" \
    --pexels-key     \"$PEXELS_API_KEY\" \
    --photo-bank-max-uses {PHOTO_BANK_MAX_USES} \
    --review-dir     \"{REVIEW_DIR}\"
"""

# Execute the command using get_ipython().system() for robustness
get_ipython().system(command)

print("🧩 Prebuild completed!")
print()
print("🎪 Expected log lines (confirm above):")
print("   Portrait pool detected at /content/Lamahat/resources/character — skipping pinned-portrait copy")
print("   Book cover directory pool detected at /content/Lamahat/resources/book_cover — skipping prebuild copy")
print("   Photo bank auto-detected at /content/Lamahat/resources/photo_bank (Path C) — if present:")
print("     photo_bank: N/59 image shots assigned from M bank photos")
print("     (curated photos become each assigned shot's chosen winner;")
print("      add --photo-bank-only to skip the web waterfall for those shots)")

INFO    phase3.prebuild  Loaded plan: 85 shots
INFO    phase3.prebuild  Image-needing shots: 53 (the rest are typography)
INFO    phase3.prebuild  Portrait pool detected at /content/Lamahat/resources/character — skipping pinned-portrait copy (pool will be used at render time)
INFO    phase3.prebuild  Book cover directory pool detected at /content/Lamahat/resources/book_cover — skipping prebuild copy (render_plan will auto-discover it)
INFO    phase3.prebuild  Photo bank auto-detected at /content/Lamahat/resources/photo_bank
INFO    phase3.prebuild  Photo bank: captioning 20 photo(s) (cached ones are free)…
INFO    phase3.sources.photo_bank  photo_bank: captioned user_arab_congress.jpeg — Group portrait of Iraqi military and political officials from the earl
INFO    phase3.sources.photo_bank  photo_bank: captioned user_arab_nationalism_flag.jpeg — Palestinian flag in three horizontal stripes of black, white, and gree
INFO    phase3.sources.photo_bank  photo_bank: captioned user_arab_rev

In [10]:
# Zip prebuild assets
import os
import zipfile

output_dir = REVIEW_DIR
zip_filename = REVIEW_FILE

# Create the zip archive
with zipfile.ZipFile(zip_filename, 'w', zipfile.ZIP_DEFLATED) as zipf:
    files_added = []
    for root, dirs, files in os.walk(output_dir):
        for file in files:
            file_path = os.path.join(root, file)
            # Add file to zip, preserving directory structure relative to 'output_dir'
            # Skip .mp3 files if any exist (though unlikely in review folder)
            if not file_path.endswith('.mp3'):
                zipf.write(file_path, os.path.relpath(file_path, output_dir))
                files_added.append(file_path)

print(f"🤐 Successfully created '{zip_filename}' containing: ")
if files_added:
    for f in files_added:
        print(f"  - {f}")
else:
    print("⚠️ No files found to zip in the "+REVIEW_DIR+" directory!")

🤐 Successfully created 'output/review.zip' containing: 
  - output/review/decisions.json
  - output/review/README.txt
  - output/review/photo_bank_assignment_raw.txt
  - output/review/shot_82_portrait/wikipedia_b.jpg
  - output/review/shot_82_portrait/wikipedia_a.jpg
  - output/review/shot_82_portrait/pexels_a.jpg
  - output/review/shot_82_portrait/pexels_b.jpg
  - output/review/shot_82_portrait/candidates.json
  - output/review/shot_82_portrait/context.txt
  - output/review/shot_82_portrait/pexels_c.jpg
  - output/review/shot_81_broll/wikimedia_b.jpg
  - output/review/shot_81_broll/wikipedia_a.jpg
  - output/review/shot_81_broll/pexels_a.jpg
  - output/review/shot_81_broll/pexels_b.jpg
  - output/review/shot_81_broll/candidates.json
  - output/review/shot_81_broll/context.txt
  - output/review/shot_81_broll/wikimedia_c.jpg
  - output/review/shot_81_broll/wikimedia_a.jpg
  - output/review/shot_81_broll/pexels_c.jpg
  - output/review/shot_17_location/wikipedia_b.jpg
  - output/review/sh

In [ ]:
# Standalone test runner
# !python verify_user_marked.py

In [ ]:
# !python verify_title_card.py --book-cover overrides/book_cover/my_book.jpg

In [ ]:
# !python diagnose_issue4.py --review-dir output/review/   # is everything in place?

In [11]:
# Condition assets
!python condition_assets.py --review-dir {REVIEW_DIR} # --sr realesrgan]; --dry-run

INFO  condition_assets  Shot 2 contain upscale 935x1200 -> 1247x1600 [upscaled]
INFO  condition_assets  Shot 4 cover   asis 2560x1706 -> 2560x1706 [ok]
INFO  condition_assets  Shot 5 cover   asis 2560x1706 -> 2560x1706 [ok]
INFO  condition_assets  Shot 7 contain upscale 935x1200 -> 1247x1600 [upscaled]
INFO  condition_assets  Shot 8 cover   asis 2560x1706 -> 2560x1706 [ok]
INFO  condition_assets  Shot 9 cover   asis 2560x1706 -> 2560x1706 [ok]
INFO  condition_assets  Shot 11 cover   asis 2560x1440 -> 2560x1440 [ok]
INFO  condition_assets  Shot 14 cover   upscale 1280x959 -> 2560x1918 [upscaled]
INFO  condition_assets  Shot 15 cover   upscale 1920x1080 -> 2560x1440 [upscaled]
INFO  condition_assets  Shot 17 cover   upscale 1376x768 -> 2560x1429 [upscaled]
INFO  condition_assets  Shot 18 toned (documentary palette, source=pexels)
INFO  condition_assets  Shot 18 cover   upscale 1880x1253 -> 2560x1706 [upscaled]
INFO  condition_assets  Shot 20 cover   asis 2560x1706 -> 2560x1706 [ok]
INFO 

In [12]:
# OPTIONAL
# Zip conditioned assets
import os
import zipfile

output_dir = REVIEW_DIR
zip_filename = CONDITIONED_ZIP_FILE

# Create the zip archive
with zipfile.ZipFile(zip_filename, 'w', zipfile.ZIP_DEFLATED) as zipf:
    files_added = []
    for root, dirs, files in os.walk(output_dir):
        for file in files:
            file_path = os.path.join(root, file)
            # Add file to zip, preserving directory structure relative to 'output_dir'
            # Skip .mp3 files if any exist (though unlikely in review folder)
            if not file_path.endswith('.mp3'):
                zipf.write(file_path, os.path.relpath(file_path, output_dir))
                files_added.append(file_path)

print(f"🤐 Successfully created '{zip_filename}' containing: ")
if files_added:
    for f in files_added:
        print(f"  - {f}")
else:
    print("⚠️ No files found to zip in the", REVIEW_DIR, "directory!")

🤐 Successfully created 'output/conditioned.zip' containing: 
  - output/review/decisions.json
  - output/review/README.txt
  - output/review/photo_bank_assignment_raw.txt
  - output/review/shot_82_portrait/wikipedia_b.jpg
  - output/review/shot_82_portrait/wikipedia_a.jpg
  - output/review/shot_82_portrait/pexels_a.jpg
  - output/review/shot_82_portrait/pexels_b.jpg
  - output/review/shot_82_portrait/candidates.json
  - output/review/shot_82_portrait/context.txt
  - output/review/shot_82_portrait/wikipedia_a.cond.jpg
  - output/review/shot_82_portrait/pexels_c.jpg
  - output/review/shot_81_broll/wikimedia_b.jpg
  - output/review/shot_81_broll/wikipedia_a.jpg
  - output/review/shot_81_broll/pexels_a.jpg
  - output/review/shot_81_broll/pexels_b.jpg
  - output/review/shot_81_broll/candidates.json
  - output/review/shot_81_broll/context.txt
  - output/review/shot_81_broll/wikimedia_c.jpg
  - output/review/shot_81_broll/wikimedia_a.jpg
  - output/review/shot_81_broll/pexels_c.jpg
  - output

In [15]:
import os
from pathlib import Path

# Ensure the parent directory for OUTPUT_FILE exists
Path(OUTPUT_FILE).parent.mkdir(parents=True, exist_ok=True)

# Construct the command parts conditionally in Python
grade_map_arg = f"--grade-map {GRADE_MAP}" if GRADE_MAP else ""
word_reveal_arg = "--word-reveal" if WORD_REVEAL else ""

# Construct the full command string
render_command = f"""
python render_plan.py \
    --plan              "{PLAN_FILE}" \
    --audio             "{AUDIO_FILE}" \
    --music             "{MUSIC_BED}" \
    --music-gain        {str(MUSIC_DB)} \
    --review-dir        "{REVIEW_DIR}" \
    --book-cover-pick   {str(BOOK_COVER_PICK)} \
    --book-cover-fit    {BOOK_COVER_FIT} \
    --book-cover-align  {BOOK_COVER_ALIGN} \
    --typography-family {TYPOGRAPHY_FAMILY} \
    --parallax \
    --typography-over-image \
    --no-captions \
    --grade             {GRADE} \
    {grade_map_arg} \
    --text-scrim        {TEXT_SCRIM} \
    --overlay-anchor    {OVERLAY_ANCHOR} \
    --title-subtitle    "{TITLE_SUBTITLE}" \
    {word_reveal_arg} \
    --caption-backplate {CAPTION_BACKPLATE} \
    --output            {OUTPUT_FILE} \
    > {LOG_FILE} 2>&1 &
"""

# Execute the command using get_ipython().system()
get_ipython().system(render_command)

print("Rendering begins ...")
print(f"  log:    {LOG_FILE}")
print(f"  output: {OUTPUT_FILE}")
print()
print("Run the next cell to tail the log until completion.")
print("Rendering could nearly take up to 40 minutes!")

Rendering begins ...
  log:    output/render.log
  output: output/final_cut_B.mp4

Run the next cell to tail the log until completion.
Rendering could nearly take up to 40 minutes!


In [16]:
# Monitor rendering progress
import time
from IPython.display import clear_output

log_path = LOG_FILE

print("Monitoring rendering progress...")
while True:
    try:
        # Read the log file contents
        try:
            with open(log_path, "r") as f:
                log_content = f.read()
        except FileNotFoundError:
            log_content = ""

        # Clear cell output and show the last 20 lines
        clear_output(wait=True)
        lines = log_content.splitlines()
        print("\n".join(lines[-20:]))

        # Check if the script's success signature is in the log
        if "Done in" in log_content or "Rendered video →" in log_content:
            print("\n✅ Rendering process completed successfully! Stopped monitoring.")
            break

        time.sleep(5)

    except KeyboardInterrupt:
        print("\n⚠️ Monitoring stopped manually. The script may still be running.")
        break


INFO  phase3.sources.decisions  Shot 81: chosen-file hit pexels_a.cond.jpg
INFO  phase3.sources  Shot 81: review-dossier hit pexels_a.cond.jpg
INFO  phase3.render  Shot 81: using fetched image from review_dossier
INFO  phase3.render  [render 72%] shot 81/85: broll
INFO  phase3.sources.decisions  Shot 82: portrait pool hit jafar_in_office.jpg (rank 8 of 11)
INFO  phase3.sources  Shot 82: review-dossier hit jafar_in_office.jpg
INFO  phase3.render  Shot 82: using fetched image from review_dossier
INFO  phase3.render  [render 73%] shot 82/85: portrait
INFO  phase3.render  [render 73%] shot 83/85: typography
INFO  phase3.sources.decisions  Shot 82: portrait pool hit jafar_in_office.jpg (rank 8 of 11)
INFO  phase3.render  [render 74%] shot 84/85: typography
INFO  phase3.render  [render 75%] shot 85/85: title_card
INFO  phase3.render  [render 80%] concat all shots
INFO  phase3.render  [render 92%] mux audio and captions
INFO  phase3.render  Color grade applied: neutral
INFO  phase3.render  Mu

In [ ]:
# !python diagnose_grade.py

In [ ]:
# !python diagnose_captions.py --plan output/re_generated_plan.json   # inspect actual ASS events

In [17]:
# Zip output files for exporting
import os
import zipfile

output_dir = OUTPUT_BASE_DIR
zip_filename = FINAL_ZIP_FILE

# Get all files in the output directory
files_to_zip = [os.path.join(output_dir, f) for f in os.listdir(output_dir) if os.path.isfile(os.path.join(output_dir, f))]

# Filter out the .mp3 file
filtered_files = [f for f in files_to_zip if not f.endswith('.mp3')]

# Create the zip archive
with zipfile.ZipFile(zip_filename, 'w', zipfile.ZIP_DEFLATED) as zipf:
    for file_path in filtered_files:
        # Add file to zip, preserving directory structure relative to 'output_dir'
        zipf.write(file_path, os.path.relpath(file_path, output_dir))

print(f"🤐 Successfully created '{zip_filename}' containing: ")
for f in filtered_files:
    print(f"  - {f}")

🤐 Successfully created 'output_files.zip' containing: 
  - output/re_generated_plan.json
  - output/conditioned.zip
  - output/planner_raw_response.txt
  - output/word_timings.json
  - output/review.zip
  - output/render.log
  - output/final_cut_B.mp4


In [18]:
# Save zipped file to Drive
from google.colab import drive
drive.mount('/content/drive')

import shutil
import os

# Define the destination directory and file path
dest_dir = DRIVE_SAVE_DIR
dest_file_path = os.path.join(dest_dir, FINAL_ZIP_FILE)

# Create the destination directory if it doesn't exist
os.makedirs(dest_dir, exist_ok=True)
print(f"Destination directory '{dest_dir}' ensured to exist.")

# --- Test write access to the directory ---
test_file = os.path.join(dest_dir, 'test_write.txt')
try:
    with open(test_file, 'w') as f:
        f.write('This is a test file.\n')
    print(f"✅ Successfully wrote test file to '{test_file}'.")
    os.remove(test_file) # Clean up the test file
    print(f"Test file '{test_file}' removed.")
except Exception as e:
    print(f"Error writing test file to '{test_file}': {e}")
    print("⚠️ It seems there might be a permissions or access issue with Google Drive.")
    # Exit or raise an error if write access fails
    raise
# ----------------------------------------

shutil.copy(FINAL_ZIP_FILE, dest_file_path)
print(f"📽️ Files are saved to Google Drive at '{dest_file_path}'.")

Mounted at /content/drive
Destination directory '/content/drive/MyDrive/_Phase3/output' ensured to exist.
✅ Successfully wrote test file to '/content/drive/MyDrive/_Phase3/output/test_write.txt'.
Test file '/content/drive/MyDrive/_Phase3/output/test_write.txt' removed.
📽️ Files are saved to Google Drive at '/content/drive/MyDrive/_Phase3/output/output_files.zip'.
